In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time

## CVM - Informações Diárias 

### 1. Verificando os arquivos

Essa logicaaa prioriza dos de M-0 e M-1, devido a atualização na base origem. Casos os dados forem de meses M-2 + serão atualizado aos domingos quando a base origem atualiza. Importante logica para controle de carga da pipeline e para FinOps.

In [0]:
BASE_URL = "https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/"

response = requests.get(BASE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# regex para pegar apenas arquivos de 2026
pattern = re.compile(r"inf_diario_fi_(\d{6})\.zip")

hoje = datetime.today()
current_year_month = hoje.strftime("%Y%m")
current_month = hoje.month
current_year = hoje.year

dia_da_semana_update = hoje.weekday() == 6 # Domingo


files = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    match = pattern.match(href)

    if match:
        file_yyyymm = match.group(1)
        file_year = int(file_yyyymm[:4])
        file_month = int(file_yyyymm[4:6])
        
        diff_month = (current_year - file_year) * 12 + (current_month - file_month)
        # ========================================== 
        #           REGRA DE ATUALIZAÇÃO
        #===========================================
        # M-0 e M-1 > atualizam diariamente
        # M-2 + > Atualizam semanalmente nos domingos 


        
        if diff_month <= 1:
            files.append(urljoin(BASE_URL, href))

        elif diff_month >= 2 and dia_da_semana_update:
            files.append(urljoin(BASE_URL, href))


print(files)

### 2. Extraindo os arquivos

In [0]:

for zip_url in files:
    print(f'Processando: {zip_url}')

    response = requests.get(zip_url)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv"):
                output_path = os.path.join("/Volumes/workspace/case_spark_cvm/raw/cvm_informe_diario/", file_name)

                with z.open(file_name) as source, open(output_path, "wb") as target:
                    target.write(source.read())

                print(f"  → Extraído: {output_path}")


### 3. Salvar em camada Bronze Particionada

Nesta etapa, realizamos a ***normalização*** das colunas. Nos arquivos recentes, foi incluída a coluna **ID_SUBCLASSE**; já nos registros de anos anteriores, essa coluna é inexistente e a nomenclatura das demais difere do padrão atual. Esse processo garante a padronização e a ordenação ***correta*** dos dados para o consumo.

In [0]:
df_temp = spark.read.csv("/Volumes/workspace/case_spark_cvm/raw/cvm_informe_diario/", sep=';', header=True)

In [0]:
display(df_temp.where(f.col("cnpj_fundo_classe") == "08.971.868/0001-40")) # "26.452.179/0001-01"

In [0]:
caminho_raw = "/Volumes/workspace/case_spark_cvm/raw/cvm_informe_diario/"

# 1. Lista todos os arquivos CSV dentro do volume
arquivos_raw = [file.path for file in dbutils.fs.ls(caminho_raw) if file.name.endswith('.csv')]

lista_dfs = []

# Ordem oficial que queremos na nossa tabela Bronze
colunas_ordem = [
    "TP_FUNDO_CLASSE", "CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE",
    "DT_COMPTC", "VL_TOTAL", "VL_QUOTA", "VL_PATRIM_LIQ",
    "CAPTC_DIA", "RESG_DIA", "NR_COTST"
]

# 2. Lê e padroniza cada arquivo
for arquivo in arquivos_raw:
    df_temp = spark.read.csv(arquivo, sep=';', header=True)
    
    # Se detectar que é o layout antigo (tem TP_FUNDO em vez de TP_FUNDO_CLASSE)
    if "TP_FUNDO" in df_temp.columns:
        df_temp = df_temp \
            .withColumnRenamed("TP_FUNDO", "TP_FUNDO_CLASSE") \
            .withColumnRenamed("CNPJ_FUNDO", "CNPJ_FUNDO_CLASSE") \
            .withColumn("ID_SUBCLASSE", f.lit(None).cast("string")) # Cria a coluna faltante como nula
            
    # Garante que todos os dataframes tenham as colunas na mesma ordem
    df_temp = df_temp.select(*colunas_ordem)
    
    lista_dfs.append(df_temp)

# 3. Une todos os DataFrames corretamente (agora todos têm as mesmas 10 colunas)
df_cvm = reduce(DataFrame.unionByName, lista_dfs)

# 4. Adiciona a data de processamento
df_cvm = df_cvm.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

# 5. Salva na camada Bronze
data_proc = int(datetime.now().strftime("%Y%m%d"))

df_cvm.write \
    .mode("overwrite") \
    .option("replaceWhere", f"data_processamento = {data_proc}") \
    .partitionBy("data_processamento") \
    .format("delta") \
    .save("/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/")

In [0]:
display(df_cvm.where(f.col("cnpj_fundo_classe") == "08.971.868/0001-40")) #"26.452.179/0001-01"

In [0]:
display(df_cvm)

In [0]:
df_cvm = spark.read\
    .option("mergeSchema", "true") \
    .option("header", True)\
    .option("delimiter", ";")\
    .csv("/Volumes/workspace/case_spark_cvm/raw/cvm_informe_diario/")

df_cvm = df_cvm.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_cvm.write \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .option("replaceWhere", f"data_processamento = {data_proc}") \
    .partitionBy("data_processamento") \
    .format("delta") \
    .save("/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/")

## CVM - Fundos Imobiliarios

### 1. Verificando os arquivos

In [0]:
BASE_URL = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/"

response = requests.get(BASE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# regex para pegar apenas arquivos de 2026      
pattern = re.compile(r"inf_mensal_fii_(\d{4})\.zip")

files = []


for link in soup.find_all("a", href=True):
    href = link["href"]
    match = pattern.match(href)
    if match:
        files.append(urljoin(BASE_URL, href))



print(files)

### 2. Extraindo os arquivos

In [0]:
for zip_url in files: 
    print(f'Processando: {zip_url}')

    response = requests.get(zip_url)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv"):
                output_path = os.path.join("/Volumes/workspace/case_spark_cvm/raw/cvm_fii/", file_name)

                with z.open(file_name) as source, open(output_path, "wb") as target:
                    target.write(source.read())

                print(f"  → Extraído: {output_path}")


### 3. Salvar em camada Bronze Particionada

In [0]:
import re
import os
from datetime import datetime
import pyspark.sql.functions as f

# Caminho RAW
RAW_PATH = "/Volumes/workspace/case_spark_cvm/raw/cvm_fii/"

# Data de processamento
data_proc = int(datetime.now().strftime(r"%Y%m%d"))

# Regex para identificar tipo do arquivo
pattern = re.compile(r"inf_mensal_fii_(ativo_passivo|complemento|geral)_\d{4}\.csv")

# Lista arquivos da pasta RAW
arquivos = dbutils.fs.ls(RAW_PATH)

# Dicionário para organizar arquivos por tipo
lista_registros = {
    "cvm_fii_ativo_passivo": [],
    "cvm_fii_complemento": [],
    "cvm_fii_geral": []
}

# Classifica os arquivos
for file in arquivos:
    nome_arquivo = os.path.basename(file.path)
    match = pattern.match(nome_arquivo)

    if match:
        tipo = match.group(1)
        if tipo == "ativo_passivo":
            lista_registros["cvm_fii_ativo_passivo"].append(file.path)
        elif tipo == "complemento":
            lista_registros["cvm_fii_complemento"].append(file.path)
        elif tipo == "geral":
            lista_registros["cvm_fii_geral"].append(file.path)



for name_path, caminhos in lista_registros.items():

    if not caminhos:
        continue

    output_path = f"/Volumes/workspace/case_spark_cvm/bronze/{name_path}/"

    print(f"\n Processamento: {name_path}")
    print(f"Arquivos: {len(caminhos)}")
    print(f"Output: {output_path}")

    # ---------------------------------------------------------
    # SOLUÇÃO PARA O SCHEMA DRIFT NO ARQUIVO GERAL
    # ---------------------------------------------------------
    if name_path == "cvm_fii_geral":
        df_final = None
        
        for path in caminhos:
            df_temp = spark.read.csv(path, sep=';', header=True)
            
            # 1. Renomeia colunas legadas (2016) para o padrão atual
            if "CNPJ_Fundo" in df_temp.columns:
                df_temp = df_temp.withColumnRenamed("CNPJ_Fundo", "CNPJ_FUNDO_CLASSE")
            
            if "Nome_Fundo" in df_temp.columns:
                df_temp = df_temp.withColumnRenamed("Nome_Fundo", "Nome_Fundo_Classe")
            
            # 2. Adiciona a coluna Tipo_Fundo_Classe caso ela não exista
            if "Tipo_Fundo_Classe" not in df_temp.columns:
                df_temp = df_temp.withColumn("Tipo_Fundo_Classe", f.lit(None).cast("string"))

            # 3. Empilha os dataframes mapeando pelo NOME da coluna, não pela posição
            if df_final is None:
                df_final = df_temp
            else:
                # allowMissingColumns=True garante que se houver mais alguma coluna nova, o código não quebre
                df_final = df_final.unionByName(df_temp, allowMissingColumns=True)
        
        df = df_final

    else:
        # Leitura padrão em lote para ativo_passivo e complemento
        df = spark.read.csv(caminhos, sep=';', header=True)
    # ---------------------------------------------------------

    # Adiciona data processamento
    df = df.withColumn(
        "data_processamento",
        f.lit(data_proc)
    )

    # Escrita em Delta
    df.write \
        .mode('overwrite') \
        .option("mergeSchema", "true") \
        .option("replaceWhere", f"data_processamento = {data_proc}") \
        .partitionBy('data_processamento') \
        .format('delta') \
        .save(output_path)

In [0]:
# # Caminho RAW
# RAW_PATH = "/Volumes/workspace/case_spark_cvm/raw/cvm_fii/"

# # Data de processamento
# data_proc = int(datetime.now().strftime(r"%Y%m%d"))

# # Regex para identificar tipo do arquivo
# pattern = re.compile(r"inf_mensal_fii_(ativo_passivo|complemento|geral)_\d{4}\.csv")

# # Lista arquivos da pasta RAW
# arquivos = dbutils.fs.ls(RAW_PATH)

# # Dicionário para organizar arquivos por tipo
# lista_registros = {
#     "cvm_fii_ativo_passivo": [],
#     "cvm_fii_complemento": [],
#     "cvm_fii_geral": []
# }

# # Classifica os arquivos
# for file in arquivos:
#     nome_arquivo = os.path.basename(file.path)

#     match = pattern.match(nome_arquivo)

#     if match:
#         tipo = match.group(1)

#         if tipo == "ativo_passivo":
#             lista_registros["cvm_fii_ativo_passivo"].append(file.path)

#         elif tipo == "complemento":
#             lista_registros["cvm_fii_complemento"].append(file.path)

#         elif tipo == "geral":
#             lista_registros["cvm_fii_geral"].append(file.path)


# # Evita rodar domingo e segunda
# hoje = datetime.today()

# if hoje.weekday() in [6, 0]:
#     print("Sem processamento nas segundas e domingos")

# else:
#     for name_path, caminhos in lista_registros.items():

#         if not caminhos:
#             continue

#         output_path = f"/Volumes/workspace/case_spark_cvm/bronze/{name_path}/"

#         print(f"\n Processamento: {name_path}")
#         print(f"Arquivos: {len(caminhos)}")
#         print(f"Output: {output_path}")

#         # Lê todos os anos juntos
#         df = spark.read.csv(caminhos, sep=';', header=True)

#         # Adiciona data processamento
#         df = df.withColumn(
#             "data_processamento",
#             f.lit(data_proc)
#         )

#         # Escrita em Delta
#         df.write \
#             .mode('overwrite') \
#             .option("replaceWhere", f"data_processamento = {data_proc}") \
#             .partitionBy('data_processamento') \
#             .format('delta') \
#             .save(output_path)

## CVM - Fundos de Investimento, Classes e Subclasses de Cotas

### 1. Verificando os arquivos

Segundo o Site da CVM a atualização dos arquivos ocorre de terça a sábado, às 08:00h, com a posição cadastral dos fundos até as 23:59h do dia anterior. Sendo assim conseguimos economizar no processamento nesses dias 


In [0]:

lista_registros = dict()
BASE_URL_DATA_CRUSE = "https://dados.cvm.gov.br/dados/FI/CAD/DADOS/registro_fundo_classe.zip"


# 0 = segunda, 6 = domingo
if hoje.weekday() in [0, 6]:
    print("Sem ingestão nas segundas e domingos")
else:
    # BAIXA OS DADOS (RAW)
    response = requests.get(BASE_URL_DATA_CRUSE)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv"):
                output_path = os.path.join("/Volumes/workspace/case_spark_cvm/raw/cvm_registros/", file_name)

                with z.open(file_name) as source, open(output_path, "wb") as target:
                    target.write(source.read())

                print(f"  → Extraído: {output_path}")
                lista_registros[output_path.split('/')[-1].split(".csv")[0]] = output_path

### 2. Salvar em camada Bronze Particionada

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

if hoje.weekday() in [6, 0]: 
    print("Sem processamento nas segundas e Domingos")

else:
    for name_path, caminho in lista_registros.items():

        output_path = f"/Volumes/workspace/case_spark_cvm/bronze/{name_path}_cvm/"
        print(f'Processamento: {name_path}')
        print(f"Output: {output_path}")

        df = spark.read.csv(caminho, sep=';', header=True)

        df = df.withColumn(
                "data_processamento",
            f.date_format(f.current_date(), "yyyyMMdd").cast("int")
        )

        df.write \
        .mode('overwrite') \
        .option("replaceWhere", f"data_processamento = {data_proc}")\
        .partitionBy('data_processamento')\
        .format('delta')\
        .save(output_path)

## Taxa SELIC

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
hoje = datetime.today()
menos_10_anos = hoje.replace(year=hoje.year - 10)
menos_10_anos = menos_10_anos.strftime(r"%d/%m/%Y")

url_bc_selic = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.1178/dados?formato=json&dataInicial={menos_10_anos}"

response = requests.get(url_bc_selic)

response.raise_for_status()
data_selic = response.json()



In [0]:
file_path_selic = "/Volumes/workspace/case_spark_cvm/raw/data_selic/"
df_selic = spark.createDataFrame(data_selic)
df_selic.write.mode('overwrite').parquet(file_path_selic)

### 2. Salvar em camada Bronze Particionada

In [0]:
df_selic_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_selic/")

df_selic_bronze = df_selic_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_selic_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_selic/")

## CDI

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
hoje = datetime.today()
menos_10_anos = hoje.replace(year=hoje.year - 10)
menos_10_anos = menos_10_anos.strftime(r"%d/%m/%Y")

#CDI (taxa diária — código 12):
url_bc_selic = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json&dataInicial={menos_10_anos}"

response = requests.get(url_bc_selic)

response.raise_for_status()
data_cdi = response.json()

In [0]:
file_path_selic = "/Volumes/workspace/case_spark_cvm/raw/data_cdi_diario/"
df_cdi = spark.createDataFrame(data_cdi)
df_cdi.write.mode('overwrite').parquet(file_path_selic)

### 2. Salvar em camada Bronze Particionada

In [0]:
df_cdi_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_cdi_diario/")

df_cdi_bronze = df_cdi_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_cdi_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_cdi_diario/")

## IPCA

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
hoje = datetime.today()
menos_10_anos = hoje.replace(year=hoje.year - 10)
menos_10_anos = menos_10_anos.strftime(r"%d/%m/%Y")

#IPCA (mensal — código 433):
url_bc_selic = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial={menos_10_anos}"

response = requests.get(url_bc_selic)

response.raise_for_status()
data_ipca = response.json()

In [0]:
file_path_selic = "/Volumes/workspace/case_spark_cvm/raw/data_ipca_mensal/"
df_ipca = spark.createDataFrame(data_ipca)
df_ipca.write.mode('overwrite').parquet(file_path_selic)

### 2. Salvar em camada Bronze Particionada

In [0]:
df_ipca_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_ipca_mensal/")

df_ipca_bronze = df_ipca_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_ipca_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_ipca_mensal/")

## IBOV

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
url = "https://query1.finance.yahoo.com/v8/finance/chart/%5EBVSP?range=10y&interval=1d"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

data = response.json()

timestamps = data["chart"]["result"][0]["timestamp"]
close = data["chart"]["result"][0]["indicators"]["quote"][0]["close"]

# criar lista de registros
records = list(zip(timestamps, close))

# criar dataframe spark
df_ibov = spark.createDataFrame(records, ["timestamp", "close"])

In [0]:
file_path_ibov = "/Volumes/workspace/case_spark_cvm/raw/data_ibov/"
df_ibov.write.mode('overwrite').parquet(file_path_ibov)

### 2. Salvar em camada Bronze Particionada

In [0]:
df_ibov_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_ibov/")

df_ibov_bronze = df_ibov_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)



In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_ibov_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_ibov/")

## IFIX

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
URL_BASE = "https://api.fintz.com.br"
HEADERS = {"X-API-Key": "chave-de-testes-api-fintz"}  # chave pública de testes
PARAMS = {
    "indice": "IFIX",
    "dataInicio": "2012-01-01",
    "ordem": "ASC"
}

response = requests.get(f"{URL_BASE}/indices/historico", headers=HEADERS, params=PARAMS)


In [0]:
if response.status_code == 200:
    df_ifix = spark.createDataFrame(response.json())
    periodo = df_ifix.select(
        f.min("data").alias("data_min"),
        f.max("data").alias("data_max")
    ).collect()[0]

    print(f"Período: {periodo['data_min']} até {periodo['data_max']}")

    print(f"Registros: {df_ifix.count()}")

    display(df_ifix.limit(5))
else:
    print(f"Erro {response.status_code}: {response.text}")

In [0]:
if response.status_code == 200:
    df_ifix = spark.createDataFrame(response.json())
    periodo = df_ifix.select(
        f.min("data").alias("data_min"),
        f.max("data").alias("data_max")
    ).collect()[0]

    print(f"Período: {periodo['data_min']} até {periodo['data_max']}")

    print(f"Registros: {df_ifix.count()}")

    display(df_ifix.limit(5))
else:
    print(f"Erro {response.status_code}: {response.text}")

In [0]:
file_path_ifix = "/Volumes/workspace/case_spark_cvm/bronze/data_ifix/"
df_ifix.write.mode('overwrite').parquet(file_path_ifix)

### 2. Salvar em camada Bronze Particionada

In [0]:

df_ifix_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_selic/")

df_ifix_bronze = df_ifix_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_ifix_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_selic/")